# Numerical edge cases, and what the warnings mean

Two questions this notebook answers. First: which degenerate, empty or otherwise
pathological inputs return a **number** rather than a `NaN`? Exact degeneracies are the usual
place a closed-form implementation divides by zero, and the Magnus expansion never forms those
denominators -- it exponentiates a matrix, and a degenerate matrix exponentiates perfectly
well.

Second, and more useful in practice: Mag$\nu$s has **nine** warning classes, and they do not
all mean the same kind of thing. Some report a bad input, some an expensive choice, and some a
condition that was not met but may not matter. Knowing which is which is the difference between
a warning you act on and one you note.

In [1]:
# The figures are set through LaTeX where one is available; where it is not,
# matplotlib's own mathtext renders the labels instead.  Same numbers either way.
import shutil

import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = shutil.which('latex') is not None

In [2]:
import warnings

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# Mag(nu)s is imported as an installed package -- from the repository root,
# 'pip install -e .' (add [plot] for magnus.plotting). No sys.path juggling.
import magnus.magnus as magnus
import magnus.oscprob as oscprob
import magnus.hamiltonians as hamiltonians
import magnus.matter as matter
import magnus.globaldefs as gd

# load_nufit_params returns exactly the six mixing parameters, ready to splat
# into any osc_prob_3nu_* call.  'NuFIT 6.1' is the package default.
OSC = gd.load_nufit_params('NuFIT 6.1', 'NO')
osc = OSC
h_vac = np.asarray(hamiltonians.hamiltonian_3nu_vacuum_energy_independent(**OSC))

ENERGY = 1.0*gd.UNIT_GEV
BASELINE = 1300.0*gd.UNIT_KM

## 1. A Hamiltonian proportional to the identity

If every eigenvalue is the same there is no relative phase, so nothing oscillates. The answer
is the identity matrix, exactly.

In [3]:
P = np.asarray(oscprob.osc_prob(np.eye(3)*3.7e-13, 0.0, BASELINE))
print(np.round(P, 12))

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 2. The trace does not matter

Adding a multiple of the identity shifts every eigenvalue equally, which multiplies the
evolution operator by an overall phase and cancels in the probability. Useful in practice: you
may drop the trace of your Hamiltonian without changing anything.

In [4]:
P_plain = np.asarray(oscprob.osc_prob(h_vac/ENERGY, 0.0, BASELINE))
shift = 3.0*np.max(np.abs(h_vac/ENERGY))     # comparable to H itself
P_shifted = np.asarray(oscprob.osc_prob(h_vac/ENERGY + shift*np.eye(3), 0.0, BASELINE))

print('max |P(H) - P(H + c*I)| = %.2e' % np.max(np.abs(P_plain - P_shifted)))

max |P(H) - P(H + c*I)| = 2.00e-15


## 3. Zero baseline, and an exactly degenerate spectrum

A zero baseline means no evolution; equal masses mean no oscillation. Both return the identity
rather than a division by zero.

In [5]:
print('L = 0:')
print(np.round(np.asarray(oscprob.osc_prob(h_vac/ENERGY, 0.0, 0.0)), 12))

h_degenerate = np.asarray(hamiltonians.hamiltonian_3nu_vacuum_energy_independent(
    **dict(OSC, D21=0.0, D31=0.0)))
print('\nD21 = D31 = 0:')
print(np.round(np.asarray(oscprob.osc_prob(h_degenerate/ENERGY, 0.0, BASELINE)), 12))

L = 0:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

D21 = D31 = 0:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 4. Approaching degeneracy

The interesting case is not the exact degeneracy but the approach to it, where a closed form
divides by a vanishing splitting. Here the probability tends smoothly to the identity and
unitarity holds at the $10^{-15}$ level throughout -- fifteen orders of magnitude of shrinking
splitting, with no special-casing anywhere.

In [6]:
print('%-12s %-18s %s' % ('scale', 'P_ee', 'max |row sum - 1|'))
print('-'*50)
for scale in (1e-3, 1e-6, 1e-9, 1e-12, 1e-15):
    h_near = np.asarray(hamiltonians.hamiltonian_3nu_vacuum_energy_independent(
        **dict(OSC, D21=scale*OSC['D21'], D31=scale*OSC['D31'])))
    P = np.asarray(oscprob.osc_prob(h_near/ENERGY, 0.0, BASELINE))
    print('%-12.0e %-18.12f %.1e'
          % (scale, P[0][0], np.max(np.abs(P.sum(axis=1) - 1.0))))

scale        P_ee               max |row sum - 1|
--------------------------------------------------
1e-03        0.999998511507     3.3e-16
1e-06        0.999999999999     2.2e-16
1e-09        1.000000000000     0.0e+00
1e-12        1.000000000000     0.0e+00
1e-15        1.000000000000     0.0e+00


## 5. Degenerate *requests*

A single slab, and no tolerance at all. Both are legitimate: `n_slabs=1` asks for one Magnus
step over the whole baseline, and `rtol=atol=None` switches the adaptive ladder off entirely.
For a constant Hamiltonian one slab is already exact, so all three agree.

In [7]:
for label, kwargs in (('default        ', {}),
                      ('n_slabs=1      ', dict(n_slabs=1)),
                      ('rtol=atol=None ', dict(rtol=None, atol=None))):
    P = np.asarray(oscprob.osc_prob(h_vac/ENERGY, 0.0, BASELINE, **kwargs))
    print('%s P_ee = %.12f' % (label, P[0][0]))

default         P_ee = 0.928948196281
n_slabs=1       P_ee = 0.928948196281
rtol=atol=None  P_ee = 0.928948196281


## 6. The nine warnings

| class | says | act on it? |
|---|---|---|
| `DensityUnitWarning` | a density is implausible for the units declared | **yes -- bad input** |
| `ScalarHamiltonianWarning` | your `H_func` takes one position at a time | yes -- costs speed only |
| `MagnusHighOrderCostWarning` | order > 6 with trapezoid/simpson is dear | your call |
| `MagnusConvergenceWarning` | a slab is wider than the sufficient condition | **often not** -- see below |
| `ToleranceNotAchievedWarning` | refinement stopped without agreeing | usually yes |
| `HybridCertificationWarning` | the adiabatic path could not certify itself | yes |
| `UnmarkedDiscontinuityWarning` | a density jump was detected, not declared | **yes -- pass `t_breakpoints`** |
| `HiddenFeatureWarning` | structure was found the sampling nearly missed | yes |
| `PhaseAveragingWarning` | `average=True` where the phase has not averaged | yes -- wrong question |

`MagnusConvergenceWarning` deserves its own sentence: it is a statement about **slab width, not
about the answer**, and it is measured to be a false alarm about three quarters of the time. It
fires in notebook 16 on a converged result. Do not read it as "this number is wrong"; read it
as "a sufficient condition was not met somewhere".

Below, each of six is provoked deliberately.

In [8]:
def provoke(label, call):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        try:
            call()
            note = ''
        except Exception as exc:
            note = '  [raised %s]' % type(exc).__name__
    names = sorted({w.category.__name__ for w in caught
                    if issubclass(w.category, Warning)})
    print('%-32s %s%s' % (label, ', '.join(names) if names else '(quiet)', note))

In [9]:
NE0, L_SCALE = gd.NUM_DENSITY_E_SUN_CENTRAL, gd.L_SCALE_SUN
PARAMS_2NU = {'sth': 0.55, 'Dm2': 7.5e-5}

# 1. a density in g/cm^3 with the flag left at its default
provoke('density under-converted', lambda: oscprob.osc_prob_3nu_matter_constant_density(
    ENERGY, BASELINE, 2.848, **OSC))

# 2. the mirror mistake: already converted, but declared as g/cm^3
provoke('density double-converted', lambda: oscprob.osc_prob_3nu_matter_constant_density(
    ENERGY, BASELINE, 100.0*gd.UNIT_G_PER_CM3, **OSC,
    density_matter_is_in_g_per_cm3=True))

# 3. an H_func that only accepts one position at a time
provoke('scalar H_func', lambda: oscprob.osc_prob(
    lambda l: h_vac/ENERGY + np.diag([float(np.asarray(l))*0.0, 0.0, 0.0]),
    0.0, BASELINE))

# 4. order 8 with a non-Gauss-Legendre integrator
provoke('order 8, simpson', lambda: magnus.magnus_expansion(
    lambda t: -1j*np.array([[0.0, 1.0], [1.0, 0.0]])*1e-13,
    0.0, 1e13, order=8, integration_method='simpson', n_tpts=20))

# 5. a density jump the caller did not declare
def step_ne(l):
    x = np.asarray(l, dtype=float)
    out = np.where(x < 0.5*L_SCALE, 0.02*NE0, 0.30*NE0)
    return out[()] if out.ndim == 0 else out

provoke('unmarked density jump', lambda: oscprob.osc_prob_matter_std_potential(
    2, step_ne, 50.0e6, 1.0*L_SCALE, PARAMS_2NU, L0=0.0,
    density_is_of_number_of_electrons=True))

# 6. asking for the averaged probability where nothing has averaged yet
provoke('average=True, few cycles', lambda: oscprob.osc_prob_3nu_vacuum(
    ENERGY, 5.0*gd.UNIT_KM, **OSC, average=True))

# ... and the same request where it genuinely has
provoke('average=True, many cycles', lambda: oscprob.osc_prob_3nu_vacuum(
    ENERGY, 5.0e4*gd.UNIT_KM, **OSC, average=True))

density under-converted          DensityUnitWarning
density double-converted         DensityUnitWarning
scalar H_func                    MagnusConvergenceWarning, ScalarHamiltonianWarning
order 8, simpson                 MagnusHighOrderCostWarning
unmarked density jump            MagnusConvergenceWarning, UnmarkedDiscontinuityWarning
average=True, few cycles         PhaseAveragingWarning
average=True, many cycles        (quiet)


The last two lines are the pattern worth internalising. `PhaseAveragingWarning` is
not about accuracy -- the returned matrix is a perfectly valid doubly stochastic probability
matrix either way. It says the *question* does not apply at that baseline, because the phase
has not averaged and no averaged expression describes it. Move far enough out and it goes
quiet.

## Summary

Nothing in section 1--5 returns a `NaN`, including exact degeneracies, a zero baseline and a
Hamiltonian with no structure at all. That is a property of the method rather than of
defensive coding: the Magnus expansion exponentiates a matrix, and never forms the
$1/(\lambda_i - \lambda_j)$ that closed forms must.

For the warnings, one rule: **`MagnusConvergenceWarning` is about slab width, everything else
is about you.** Measured false-alarm rates for each are in `implementation_details.rst`.

Notebook 21 takes the tolerance warnings further, and it is the one to read next if you have
ever taken `rtol` for an error bound.

---

**Previous:** [Bring your own Hamiltonian](19_magnus_custom_hamiltonian.ipynb)  
**Next:** [What rtol and atol promise](21_magnus_what_tolerance_means.ipynb) --- a stopping criterion, not an error bound  
[API reference](https://mbustama.github.io/Magnus/functions.html) &middot; [Implementation details](https://mbustama.github.io/Magnus/implementation_details.html) &middot; [All notebooks](.)